# 🚶 Walking-Tour Guide — Colab runner

Runs the **tools**, the **agent**, and the **Streamlit app** for the final project.
Run the cells top to bottom.

- Tools & agent → print inline, no tunnel needed.
- The Streamlit app → exposed with a free **cloudflared** tunnel (no signup).

> For the graded **live demo, running locally is smoother** — see `RUN_GUIDE.md`.


## 1. Get the code into Colab
Upload `DS-TravelAi-main.zip` when prompted (or swap in a `git clone` of your repo).


In [ ]:
from google.colab import files
import zipfile, os

up = files.upload()                      # pick DS-TravelAi-main.zip
zip_name = next(iter(up))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('.')

# folder name inside the zip
%cd DS-TravelAi-main
print('Now in:', os.getcwd())

# --- OR clone instead of uploading ---
# !git clone https://github.com/<you>/DS-TravelAi.git
# %cd DS-TravelAi

## 2. Install dependencies
Installs the agent, tools, and the app (streamlit, folium, streamlit-folium).


In [ ]:
!pip install -r requirements.txt -q
print('deps installed')

## 3. Add your OpenAI key
Needed only for the **agent** and the **real app** (the tools work without it).
Add a Colab secret named `OPENAI_API_KEY` via the 🔑 panel on the left, then run this.


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('key loaded from Colab Secrets')
except Exception:
    import getpass
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
    print('key entered manually')

## 4a. Tools only — no key, no tunnel
Geocode an address and fetch + order real nearby places from OpenStreetMap.


In [ ]:
from tools import get_nearby_places, order_walking_route, validate_request

ADDRESS = 'Allenby 29, Tel Aviv'
INTERESTS = ['architecture', 'cafe', 'art']

ok, msg = validate_request(ADDRESS, INTERESTS)
print(msg or 'valid ✔')
places = get_nearby_places(ADDRESS, INTERESTS)
for n, p in enumerate(order_walking_route(places), 1):
    print(f'{n}. {p.name}  ({p.category})  {p.lat:.5f},{p.lon:.5f}')

## 4b. Full agent — needs the key from step 3
Returns the JSON tour the app renders: `{intro, stops:[{name,category,lat,lon,narration}]}`.


In [ ]:
import json
from agent.agent import run_agent   # imported AFTER the key is set

raw = run_agent('Allenby 29, Tel Aviv', ['architecture', 'cafe', 'art'])
tour = json.loads(raw) if isinstance(raw, str) else raw
print(tour.get('intro', ''))
for n, s in enumerate(tour.get('stops', []), 1):
    print(f"{n}. {s['name']} ({s['category']})")
    print('   ', s['narration'])

## 5. The Streamlit app + map (public tunnel)
Starts the app and opens a `…trycloudflare.com` link. Click it to use the full UI.

Tip: to demo without a key, the app also has a built-in sample — set
`os.environ['USE_FAKE_AGENT']='1'` before launching.


In [ ]:
import subprocess, time, re, pathlib

# download the cloudflared binary (once)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared

# start the app in the background (inherits OPENAI_API_KEY from step 3)
subprocess.Popen(['streamlit', 'run', 'app.py',
                  '--server.port', '8501', '--server.headless', 'true'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)

# open the tunnel and read its public URL
log = open('cf.log', 'w')
subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
                 stdout=log, stderr=log)
url = None
for _ in range(30):
    time.sleep(1)
    m = re.search(r'https://[-\w]+\.trycloudflare\.com', pathlib.Path('cf.log').read_text())
    if m:
        url = m.group(0); break
print('👉 Open your app here:', url or 'not ready — re-run this cell')

---
If the tunnel link stops working, the Colab VM idled out — re-run cell 5,
or run locally (`streamlit run app.py`) per `RUN_GUIDE.md`.
